In [4]:
! pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=9b53f73db289d5245878f194e2648e07f497029493a9ca30219d8e3ff7fcfa1f
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta


Model A: win rate 39%

In [26]:
"""
Model A - Dự đoán burst/breakout/trend acceleration trong 14 phiên cho S&P 100
Đã sửa lỗi MultiIndex, backtest, predict và các vấn đề về index.
"""

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
from scipy.stats import spearmanr
import ta
import requests

warnings.filterwarnings('ignore')

# ==================== 2. Chu?n h?a d? li?u gi? v? daily OHLCV ====================
def _resample_one_symbol_to_daily(df):
    """G?p d? li?u intraday/1m th?nh m?t bar OHLCV theo ng?y giao d?ch."""
    df = df.copy()
    if 'Datetime' in df.columns:
        df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True, errors='coerce').dt.tz_localize(None)
        df = df.dropna(subset=['Datetime']).set_index('Datetime')
    else:
        df.index = pd.to_datetime(df.index, utc=True, errors='coerce').tz_localize(None)
        df = df[~df.index.isna()]

    df = df.sort_index()
    agg = {}
    if 'Open' in df.columns:
        agg['Open'] = 'first'
    if 'High' in df.columns:
        agg['High'] = 'max'
    if 'Low' in df.columns:
        agg['Low'] = 'min'
    if 'Close' in df.columns:
        agg['Close'] = 'last'
    if 'Volume' in df.columns:
        agg['Volume'] = 'sum'
    if 'Dividends' in df.columns:
        agg['Dividends'] = 'sum'
    if 'Stock Splits' in df.columns:
        agg['Stock Splits'] = 'sum'

    daily = df.resample('1D').agg(agg)
    required = [col for col in ['Open', 'High', 'Low', 'Close'] if col in daily.columns]
    if required:
        daily = daily.dropna(subset=required)
    return daily


def normalize_to_daily_ohlcv(data):
    """??m b?o m?i feature/label trong notebook ???c t?nh tr?n daily bars, kh?ng ph?i 1 ph?t."""
    if isinstance(data.columns, pd.MultiIndex):
        frames = {}
        for ticker in data.columns.get_level_values(0).unique():
            symbol_df = data[ticker].dropna(how='all')
            daily = _resample_one_symbol_to_daily(symbol_df)
            if not daily.empty:
                frames[ticker] = daily
        return pd.concat(frames, axis=1) if frames else data.iloc[0:0]
    return _resample_one_symbol_to_daily(data)


# ==================== 1. Lấy danh sách S&P 100 ====================
def get_sp100_tickers():
    url = 'https://en.wikipedia.org/wiki/S%26P_100'
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        tables = pd.read_html(response.text)
        
        df = next((t for t in tables if 'Symbol' in t.columns), None)
        if df is None: raise ValueError("Không tìm thấy cột 'Symbol'")

        tickers = [str(t).replace('.', '-').strip() for t in df['Symbol'].tolist()]
        if 'SPY' not in tickers: tickers.append('SPY')
        print(f"Đã lấy thành công {len(tickers)} mã (bao gồm SPY).")
        return tickers
    except Exception as e:
        print(f"Lỗi khi lấy danh sách: {e}")
        return ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'SPY']

# ==================== 2. Tải dữ liệu giá ====================
def download_data(tickers, start_date, end_date, interval='1d'):
    # Feature/label c?a model l? theo phi?n/ng?y; n?u nh?n intraday th? g?p v? daily OHLCV.
    data = yf.download(tickers, start=start_date, end=end_date, interval=interval, group_by='ticker', auto_adjust=True)
    return normalize_to_daily_ohlcv(data)

# ==================== 3. Tính label (forward 14D max return) ====================
def compute_label(close_prices, horizon=14):
    # Chỉ tính label khi có đủ horizon ngày trong tương lai
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=horizon)
    future_max = close_prices.shift(-1).rolling(window=indexer, min_periods=horizon).max()
    return (future_max - close_prices) / close_prices

# ==================== 4. Feature engineering ====================
def compute_return_dynamics(close, lookbacks=[1,3,5,10,14,20,21]):
    df = pd.DataFrame(index=close.index)
    for lb in lookbacks:
        df[f'r{lb}'] = close.pct_change(lb)
        
    df['cumret_20'] = close.pct_change(20)
    
    roll_std_20 = df['r20'].rolling(20).std().replace(0, np.nan)
    df['return_z_20'] = (df['r20'] - df['r20'].rolling(20).mean()) / roll_std_20
    df['gap_return'] = close.pct_change(1)
    return df

def compute_clv(high, low, close):
    diff = (high - low).replace(0, np.nan)
    return (close - low) / diff - 0.5

def compute_trend_momentum(close):
    df = pd.DataFrame(index=close.index)
    ema20 = close.ewm(span=20, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()
    
    df['EMA20_50_spread'] = (ema20 - ema50) / close.replace(0, np.nan)
    df['EMA50_200_spread'] = (ema50 - ema200) / close.replace(0, np.nan)
    df['EMA20_slope'] = ema20.diff(5) / 5
    df['EMA50_slope'] = ema50.diff(5) / 5
    
    df['MACD_hist'] = ta.trend.macd_diff(close)
    df['RSI14'] = ta.momentum.rsi(close, window=14)
    df['ROC10'] = ta.momentum.roc(close, window=10)
    df['ROC20'] = ta.momentum.roc(close, window=20)
    return df

def compute_breakout_positioning(close, high, low):
    df = pd.DataFrame(index=close.index)
    close_safe = close.replace(0, np.nan)
    df['dist_20d_high'] = (high.rolling(20).max() - close) / close_safe
    df['dist_55d_high'] = (high.rolling(55).max() - close) / close_safe
    df['dist_52w_high'] = (high.rolling(252).max() - close) / close_safe
    
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    df['BB_%B'] = bb.bollinger_pband()
    df['BB_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / close_safe
    
    ema20 = close.ewm(span=20).mean()
    ema50 = close.ewm(span=50).mean()
    df['dist_ema20'] = (close - ema20) / close_safe
    df['dist_ema50'] = (close - ema50) / close_safe
    return df

def compute_volatility(close, high, low, volume):
    df = pd.DataFrame(index=close.index)
    df['ATR14_ratio'] = ta.volatility.average_true_range(high, low, close, window=14) / close.replace(0, np.nan)
    df['vol20'] = close.pct_change().rolling(20).std()
    
    returns = close.pct_change()
    df['downside_vol20'] = returns.where(returns < 0, 0).rolling(20).std()
    
    cummax = close.expanding().max().replace(0, np.nan)
    df['maxdd20'] = ((close - cummax) / cummax).rolling(20).min() * (-1)
    return df

def compute_volume_flow(close, volume, high, low):
    df = pd.DataFrame(index=close.index)
    avg_vol = volume.rolling(20).mean().replace(0, np.nan)
    df['RVOL20'] = volume / avg_vol
    df['OBV_slope'] = ta.volume.on_balance_volume(close, volume).diff(5) / 5
    df['MFI14'] = ta.volume.money_flow_index(high, low, close, volume, window=14)
    df['dollar_volume_log'] = np.log((close * volume).replace(0, 1))
    df['volume_slope_20'] = volume.diff(5) / 5
    return df

def compute_market_context(close, spy_close):
    df = pd.DataFrame(index=close.index)
    rel_strength = close / spy_close.replace(0, np.nan)
    df['RS_vs_SPY_14'] = rel_strength.pct_change(14)
    df['RS_vs_SPY_30'] = rel_strength.pct_change(30)
    
    returns = close.pct_change()
    spy_returns = spy_close.pct_change()
    df['beta_60D'] = returns.rolling(60).cov(spy_returns) / spy_returns.rolling(60).var().replace(0, np.nan)
    df['SPY_above_EMA50'] = (spy_close > spy_close.ewm(span=50).mean()).astype(int)
    df['SPY_20d_return'] = spy_close.pct_change(20)
    return df

def build_features(close, high, low, volume, spy_close):
    f_ret = compute_return_dynamics(close)
    f_clv = pd.DataFrame({'CLV': compute_clv(high, low, close)}, index=close.index)
    f_trend = compute_trend_momentum(close)
    f_adx = pd.DataFrame({'ADX14': ta.trend.adx(high, low, close, window=14)}, index=close.index)
    f_break = compute_breakout_positioning(close, high, low)
    f_vol = compute_volatility(close, high, low, volume)
    f_flow = compute_volume_flow(close, volume, high, low)
    f_market = compute_market_context(close, spy_close)
    
    feat = pd.concat([f_ret, f_clv, f_trend, f_adx, f_break, f_vol, f_flow, f_market], axis=1)
    feat['sector_percentile_20d'] = 0.5   # placeholder
    
    return feat.replace([np.inf, -np.inf], np.nan)

# ==================== 5. Huấn luyện và đánh giá (sửa lỗi MultiIndex) ====================
def train_evaluate(X_train, y_train, X_test, y_test, params):
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)
    
    spearman_corr, _ = spearmanr(y_test, y_pred)
    
    # Tạo DataFrame kết quả với MultiIndex giữ nguyên
    results = pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred}, index=X_test.index)
    
    # Tính daily hit rate (top decile)
    daily_hits = []
    # Lấy các ngày duy nhất từ level 0 của MultiIndex
    unique_dates = results.index.get_level_values(0).unique()
    for date in unique_dates:
        # Lấy tất cả ticker của ngày đó
        group = results.xs(date, level=0)
        if len(group) < 10:
            continue
        group = group.sort_values('y_pred', ascending=False)
        top_n = max(1, int(0.1 * len(group)))
        top_group = group.head(top_n)
        daily_hits.append((top_group['y_true'] > 0).mean())
    
    print(f"Spearman correlation: {spearman_corr:.4f}")
    print(f"Daily Top Decile Hit Rate: {np.mean(daily_hits):.4f}")
    return model

# ==================== 6. Backtest thực tế (top N mỗi ngày) ====================
def backtest_topN_strategy(model, X_test, prices_dict, top_n=10, horizon=14):
    print("\nRunning realistic backtest (Top N each day)...")
    
    results = []
    # Lấy các ngày duy nhất từ level 0 của MultiIndex
    unique_dates = sorted(X_test.index.get_level_values(0).unique())
    
    for date in unique_dates:
        # Lấy tất cả ticker của ngày đó
        X_day = X_test.xs(date, level=0)
        # Nếu chỉ có 1 ticker, xs trả về Series -> chuyển thành DataFrame 1 dòng
        if isinstance(X_day, pd.Series):
            X_day = X_day.to_frame().T
        
        preds = model.predict(X_day)
        tickers_day = X_day.index
        
        df_pred = pd.DataFrame({
            'Ticker': tickers_day,
            'Pred': preds
        }).sort_values('Pred', ascending=False).head(top_n)
        
        daily_returns = []
        for t in df_pred['Ticker']:
            try:
                price_series = prices_dict[t]['Close']
                # Lấy giá đóng cửa ngày đó
                entry_price = price_series.loc[date]
                # 14 ngày tiếp theo (từ date+1 đến date+horizon)
                future_window = price_series.loc[date:].iloc[1:horizon+1]
                if len(future_window) == 0:
                    continue
                exit_price = future_window.max()
                ret = (exit_price - entry_price) / entry_price
                daily_returns.append(ret)
            except Exception as e:
                continue
        
        if daily_returns:
            results.append(np.mean(daily_returns))
    
    results = pd.Series(results)
    print(f"Avg 14D return per batch: {results.mean()*100:.2f}%")
    print(f"Win rate: {(results>0).mean()*100:.2f}%")
    print(f"Sharpe (annualized approx): {results.mean()/results.std() * np.sqrt(252/14):.2f}")
    return results

# ==================== 7. Dự đoán cho ngày hiện tại (phiên bản sửa lỗi spy_close) ====================
def predict_current(model, tickers):
    print("\nDownloading latest data for prediction...")
    new_data = download_data(tickers, "2023-01-01", datetime.today().strftime('%Y-%m-%d'))
    
    # Lấy SPY close an toàn
    if 'SPY' in new_data.columns.levels[0]:
        spy_close_new = new_data['SPY']['Close']
    else:
        # Fallback: lấy cột đầu tiên (hy vọng là SPY? Nhưng cẩn thận)
        spy_close_new = new_data.iloc[:, 0] if isinstance(new_data, pd.DataFrame) else new_data['Close']
        print("Warning: SPY not found, using first column as market proxy.")
    
    latest_features = []
    latest_tickers = []
    last_date_global = None
    
    for t in tickers:
        if t == 'SPY' or t not in new_data.columns.levels[0]:
            continue
        
        df_t = new_data[t].dropna()
        if len(df_t) < 260:
            continue
        
        close = df_t['Close']
        high = df_t['High']
        low = df_t['Low']
        volume = df_t['Volume']
        feats = build_features(close, high, low, volume, spy_close_new)
        
        feats = feats.replace([np.inf, -np.inf], np.nan).dropna()
        if feats.empty:
            continue
        
        last_row = feats.iloc[-1:]
        last_date_global = last_row.index[0]
        latest_features.append(last_row)
        latest_tickers.append(t)
    
    X_latest = pd.concat(latest_features, axis=0)
    X_latest.index = latest_tickers
    
    print(f"\nPrediction as of date (last available): {last_date_global.date()}")
    print(f"Predicting for {len(X_latest)} tickers...")
    
    preds = model.predict(X_latest)
    result = pd.DataFrame({
        'Ticker': X_latest.index,
        'Predicted_14D_UpMove_%': preds * 100
    }).sort_values('Predicted_14D_UpMove_%', ascending=False)
    
    print("\nTop 20 burst/breakout candidates:")
    print(result.head(20))
    return result

# ==================== 8. Main ====================
if __name__ == "__main__":
    start_date, end_date = "2019-01-01", "2024-12-31"
    train_end, test_start = "2022-12-31", "2023-01-01"
    
    print("Loading S&P 100 tickers...")
    tickers = get_sp100_tickers()
    
    print("Downloading price data...")
    data = download_data(tickers, start_date, end_date)
    
    # Lấy SPY close an toàn
    if 'SPY' in data.columns.levels[0]:
        spy_close = data['SPY']['Close']
    else:
        spy_close = data.iloc[:, 0]  # fallback
        print("Warning: SPY not found, using first column as market proxy.")
    
    all_x, all_y = [], []
    
    print("Building feature dataset...")
    for t in tickers:
        if t == 'SPY' or t not in data.columns.levels[0]:
            continue
        
        df_t = data[t].dropna()
        if len(df_t) < 50:
            continue
        
        close = df_t['Close']
        high = df_t['High']
        low = df_t['Low']
        volume = df_t['Volume']
        label = compute_label(close, horizon=14)
        feats = build_features(close, high, low, volume, spy_close)
        
        common_idx = feats.index.intersection(label.index)
        feats, y = feats.loc[common_idx], label.loc[common_idx]
        
        valid = ~(feats.isna().any(axis=1) | y.isna())
        feats_valid = feats[valid].copy()
        feats_valid['Ticker'] = t
        feats_valid.set_index('Ticker', append=True, inplace=True)
        
        y_valid = y[valid].copy()
        y_valid.index = feats_valid.index
        
        all_x.append(feats_valid)
        all_y.append(y_valid)
    
    X = pd.concat(all_x, axis=0)
    y = pd.concat(all_y, axis=0)
    
    # Phân chia train/test dựa trên ngày (level 0)
    train_mask = X.index.get_level_values(0) <= pd.to_datetime(train_end)
    test_mask = X.index.get_level_values(0) >= pd.to_datetime(test_start)
    
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
    
    params = {
        "n_estimators": 100,      # Giảm để chạy nhanh, có thể tăng lên 1200
        "max_depth": 6,
        "learning_rate": 0.02,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": 42
    }
    
    print("Training model...")
    model = train_evaluate(X_train, y_train, X_test, y_test, params)
    
    # Chuẩn bị dict giá cho backtest
    prices_dict = {}
    for t in tickers:
        if t in data.columns.levels[0]:
            prices_dict[t] = data[t]
    
    # Backtest trên test set
    bt_results = backtest_topN_strategy(model, X_test, prices_dict, top_n=10, horizon=14)
    
    # Dự đoán cho ngày hiện tại
    pred_today = predict_current(model, tickers)
    
    # Feature importance
    imp_df = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
    print("\nTop 10 features:")
    print(imp_df.head(10))
    
    # ===== Dự đoán cho giai đoạn 2025-2026 (nếu có dữ liệu mới) =====
    print("\n" + "="*60)
    print("PREDICTION FOR 2025-2026 (latest available data)")
    print("="*60)
    # Đơn giản là gọi lại predict_current - nó sẽ tải đến ngày hôm nay
    # Nếu muốn force đến 2026, có thể tải dữ liệu đến "2026-12-31"
    # Nhưng yfinance không lấy được dữ liệu tương lai. Vậy chỉ lấy đến hiện tại.
    pred_2025 = predict_current(model, tickers)


    import os
    
    # Tạo thư mục lưu kết quả nếu chưa có
    output_dir = "backtest_results_2026"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # ==================== 9. BACKTEST & EXPORT CSV theo ngay ====================
    print("\n" + "="*80)
    print("BACKTEST: EVERY SESSION (STEP=1) & EXPORTING SINGLE CSV WITH WIN LABEL")
    print("="*80)
    
    all_trading_days = spy_close_2026.index[spy_close_2026.index >= '2026-01-01']
    all_predictions = []   # List để gộp tất cả dự đoán từ các ngày
    
    step = 1   # Dự đoán mỗi phiên
    
    for i in range(0, len(all_trading_days), step):
        pred_date = all_trading_days[i]
        
        if i + 14 >= len(all_trading_days):
            break
    
        print(f"\nProcessing Date: {pred_date.date()}...")
        
        X_pred = prepare_prediction_data_for_date(pred_date, tickers, data_2026, spy_close_2026)
        
        if X_pred is None or len(X_pred) < 10:
            continue
    
        # 1. Dự đoán
        preds = model.predict(X_pred)
        df_step = pd.DataFrame({
            'Ticker': X_pred.index, 
            'Predicted_Score': preds
        }).sort_values('Predicted_Score', ascending=False).head(10)
    
        # 2. Tính toán thực tế (Max Price & Drawdown)
        detailed_records = []
        
        for t in df_step['Ticker']:
            try:
                price_series = data_2026[t]['Close']
                start_idx = price_series.index.get_loc(pred_date)
                
                entry_price = price_series.iloc[start_idx]
                future_window = price_series.iloc[start_idx + 1 : start_idx + 15]
                
                if len(future_window) < 1:
                    continue
    
                actual_max = future_window.max()
                actual_min = future_window.min()
                
                max_return = (actual_max - entry_price) / entry_price
                drawdown = (actual_min - entry_price) / entry_price
                
                # Tính label win: 1 nếu max_return > 0 và drawdown > -5%
                win = 1 if (max_return > 0 and drawdown > -0.05) else 0
                
                detailed_records.append({
                    'Date': pred_date.date(),                      # Thêm cột ngày dự đoán
                    'Ticker': t,
                    'Entry_Price': round(entry_price, 2),
                    'Predicted_Score': round(preds[X_pred.index.get_loc(t)], 4),
                    'Max_Price_14D': round(actual_max, 2),
                    'Max_Return_%': round(max_return * 100, 2),
                    'Max_Drawdown_%': round(drawdown * 100, 2),
                    'Win': win                                    # Nhãn thắng/thua
                })
            except:
                continue
        
        # Gộp các dòng của ngày này vào danh sách tổng
        all_predictions.extend(detailed_records)
    
    # Xuất tất cả vào một file CSV duy nhất
    if all_predictions:
        df_final = pd.DataFrame(all_predictions)
        # Sắp xếp theo ngày và điểm dự đoán giảm dần (tuỳ chọn)
        df_final = df_final.sort_values(['Date', 'Predicted_Score'], ascending=[True, False])
        
        csv_path = os.path.join(output_dir, "all_predictions.csv")
        df_final.to_csv(csv_path, index=False)
        
        # Tính tỷ lệ win
        total_predictions = len(df_final)
        wins = df_final['Win'].sum()
        win_rate = wins / total_predictions if total_predictions > 0 else 0
        
        print("\n" + "="*80)
        print(f"ĐÃ XUẤT FILE: {csv_path}")
        print(f"Tổng số dự đoán: {total_predictions}")
        print(f"Số lần WIN: {wins}")
        print(f"TỶ LỆ WIN: {win_rate:.2%}")
    else:
        print("\nKhông có dự đoán nào được thực hiện.")
    
    print("="*80)

Loading S&P 100 tickers...
Đã lấy thành công 102 mã (bao gồm SPY).


[*********************100%***********************]  102 of 102 completed


Building feature dataset...
Train size: (75171, 42), Test size: (48700, 42)
Training model...
Spearman correlation: 0.2334
Daily Top Decile Hit Rate: 0.8854

Running realistic backtest (Top N each day)...


[                       0%                       ]

Avg 14D return per batch: 7.99%
Win rate: 98.56%
Sharpe (annualized approx): 7.25



[*********************100%***********************]  102 of 102 completed
[                       0%                       ]


Prediction as of date (last available): 2026-05-08
Predicting for 101 tickers...

Top 20 burst/breakout candidates:
    Ticker  Predicted_14D_UpMove_%
47    INTC               12.824093
72     NOW                9.908490
48    INTU                8.959271
68      MU                8.815157
3      ACN                8.722583
26     CRM                8.719207
4     ADBE                7.740559
9     AMZN                7.471466
71     NKE                6.383970
6      AMD                6.200065
78    PLTR                6.147887
38     GEV                6.059700
57    LRCX                5.820818
100    XOM                5.740724
24     COP                5.635968
5     AMAT                5.504043
2      ABT                5.206761
80    QCOM                5.204285
15    BKNG                5.184131
22   CMCSA                5.136062

Top 10 features:
             feature  importance
27       ATR14_ratio    0.262429
19             ADX14    0.043710
12  EMA50_200_spread    0.04093

[*********************100%***********************]  102 of 102 completed



Prediction as of date (last available): 2026-05-08
Predicting for 101 tickers...

Top 20 burst/breakout candidates:
    Ticker  Predicted_14D_UpMove_%
47    INTC               12.824093
72     NOW                9.908490
48    INTU                8.959271
68      MU                8.815157
3      ACN                8.722583
26     CRM                8.719207
4     ADBE                7.740559
9     AMZN                7.471466
71     NKE                6.383970
6      AMD                6.200065
78    PLTR                6.147887
38     GEV                6.059700
57    LRCX                5.820818
100    XOM                5.740724
24     COP                5.635968
5     AMAT                5.504043
2      ABT                5.206761
80    QCOM                5.204285
15    BKNG                5.184131
22   CMCSA                5.136062

BACKTEST: EVERY SESSION (STEP=1) & EXPORTING SINGLE CSV WITH WIN LABEL

Processing Date: 2026-01-02...

Processing Date: 2026-01-05...

Processing 

Model A+C: Win rate 68%

In [6]:
"""
Model A (Upside) + Model C (Risk) cho S&P 100
- Model A: dự đoán max return 14 ngày (regression)
- Model C: dự đoán xác suất drawdown >=6% (classification)
- Kết hợp: FinalScore = Pred_A * (1 - Prob_C)
"""

import numpy as np
import pandas as pd
import yfinance as yf
import warnings
from datetime import datetime, timedelta
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb
from scipy.stats import spearmanr
import ta
import requests
import os

warnings.filterwarnings('ignore')

# ==================== 2. Chu?n h?a d? li?u gi? v? daily OHLCV ====================
def _resample_one_symbol_to_daily(df):
    """G?p d? li?u intraday/1m th?nh m?t bar OHLCV theo ng?y giao d?ch."""
    df = df.copy()
    if 'Datetime' in df.columns:
        df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True, errors='coerce').dt.tz_localize(None)
        df = df.dropna(subset=['Datetime']).set_index('Datetime')
    else:
        df.index = pd.to_datetime(df.index, utc=True, errors='coerce').tz_localize(None)
        df = df[~df.index.isna()]

    df = df.sort_index()
    agg = {}
    if 'Open' in df.columns:
        agg['Open'] = 'first'
    if 'High' in df.columns:
        agg['High'] = 'max'
    if 'Low' in df.columns:
        agg['Low'] = 'min'
    if 'Close' in df.columns:
        agg['Close'] = 'last'
    if 'Volume' in df.columns:
        agg['Volume'] = 'sum'
    if 'Dividends' in df.columns:
        agg['Dividends'] = 'sum'
    if 'Stock Splits' in df.columns:
        agg['Stock Splits'] = 'sum'

    daily = df.resample('1D').agg(agg)
    required = [col for col in ['Open', 'High', 'Low', 'Close'] if col in daily.columns]
    if required:
        daily = daily.dropna(subset=required)
    return daily


def normalize_to_daily_ohlcv(data):
    """??m b?o m?i feature/label trong notebook ???c t?nh tr?n daily bars, kh?ng ph?i 1 ph?t."""
    if isinstance(data.columns, pd.MultiIndex):
        frames = {}
        for ticker in data.columns.get_level_values(0).unique():
            symbol_df = data[ticker].dropna(how='all')
            daily = _resample_one_symbol_to_daily(symbol_df)
            if not daily.empty:
                frames[ticker] = daily
        return pd.concat(frames, axis=1) if frames else data.iloc[0:0]
    return _resample_one_symbol_to_daily(data)


# ==================== 1. Lấy danh sách S&P 100 ====================
def get_sp100_tickers():
    url = 'https://en.wikipedia.org/wiki/S%26P_100'
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        response = requests.get(url, headers=headers)
        tables = pd.read_html(response.text)
        df = next((t for t in tables if 'Symbol' in t.columns), None)
        if df is None:
            raise ValueError("Không tìm thấy cột 'Symbol'")
        tickers = [str(t).replace('.', '-').strip() for t in df['Symbol'].tolist()]
        if 'SPY' not in tickers:
            tickers.append('SPY')
        print(f"Lấy {len(tickers)} mã S&P 100 (kèm SPY).")
        return tickers
    except Exception as e:
        print(f"Lỗi: {e}, dùng danh sách mặc định")
        return ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'SPY']

# ==================== 2. Tải dữ liệu ====================
def download_data(tickers, start_date, end_date, interval='1d'):
    # Feature/label c?a model l? theo phi?n/ng?y; n?u nh?n intraday th? g?p v? daily OHLCV.
    data = yf.download(tickers, start=start_date, end=end_date, interval=interval, group_by='ticker', auto_adjust=True)
    return normalize_to_daily_ohlcv(data)

# ==================== 3. Label cho Model A (upside) ====================
def compute_label_upside(close_prices, horizon=14):
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=horizon)
    future_max = close_prices.shift(-1).rolling(window=indexer, min_periods=horizon).max()
    return (future_max - close_prices) / close_prices

# ==================== 4. Label cho Model C (risk) ====================
def compute_label_risk(close_prices, horizon=14, drawdown_threshold=0.06):
    """
    Nhãn 1 nếu có drawdown >= threshold trong horizon ngày tới
    """
    future_min = close_prices.shift(-1).rolling(window=horizon, min_periods=horizon).min()
    drawdown = (future_min - close_prices) / close_prices
    return (drawdown <= -drawdown_threshold).astype(int)

# ==================== 5. Feature engineering cho cả 2 model ====================
def compute_return_dynamics(close, lookbacks=[1,3,5,10,14,20,21]):
    df = pd.DataFrame(index=close.index)
    for lb in lookbacks:
        df[f'r{lb}'] = close.pct_change(lb)
    df['cumret_20'] = close.pct_change(20)
    roll_std_20 = df['r20'].rolling(20).std().replace(0, np.nan)
    df['return_z_20'] = (df['r20'] - df['r20'].rolling(20).mean()) / roll_std_20
    df['gap_return'] = close.pct_change(1)
    return df

def compute_clv(high, low, close):
    diff = (high - low).replace(0, np.nan)
    return (close - low) / diff - 0.5

def compute_trend_momentum(close):
    df = pd.DataFrame(index=close.index)
    ema20 = close.ewm(span=20, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()
    df['EMA20_50_spread'] = (ema20 - ema50) / close.replace(0, np.nan)
    df['EMA50_200_spread'] = (ema50 - ema200) / close.replace(0, np.nan)
    df['EMA20_slope'] = ema20.diff(5) / 5
    df['EMA50_slope'] = ema50.diff(5) / 5
    df['MACD_hist'] = ta.trend.macd_diff(close)
    df['RSI14'] = ta.momentum.rsi(close, window=14)
    df['ROC10'] = ta.momentum.roc(close, window=10)
    df['ROC20'] = ta.momentum.roc(close, window=20)
    return df

def compute_breakout_positioning(close, high, low):
    df = pd.DataFrame(index=close.index)
    close_safe = close.replace(0, np.nan)
    df['dist_20d_high'] = (high.rolling(20).max() - close) / close_safe
    df['dist_55d_high'] = (high.rolling(55).max() - close) / close_safe
    df['dist_52w_high'] = (high.rolling(252).max() - close) / close_safe
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    df['BB_%B'] = bb.bollinger_pband()
    df['BB_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / close_safe
    ema20 = close.ewm(span=20).mean()
    ema50 = close.ewm(span=50).mean()
    df['dist_ema20'] = (close - ema20) / close_safe
    df['dist_ema50'] = (close - ema50) / close_safe
    return df

def compute_volatility(close, high, low, volume):
    df = pd.DataFrame(index=close.index)
    df['ATR14_ratio'] = ta.volatility.average_true_range(high, low, close, window=14) / close.replace(0, np.nan)
    df['vol20'] = close.pct_change().rolling(20).std()
    returns = close.pct_change()
    df['downside_vol20'] = returns.where(returns < 0, 0).rolling(20).std()
    cummax = close.expanding().max().replace(0, np.nan)
    df['maxdd20'] = ((close - cummax) / cummax).rolling(20).min() * (-1)
    
    # Thêm các feature đặc biệt cho Risk Model
    df['realized_vol_10'] = returns.rolling(10).std()
    df['realized_vol_20'] = returns.rolling(20).std()
    df['vol_ratio_20_60'] = df['realized_vol_20'] / returns.rolling(60).std().replace(0, np.nan)
    df['true_range_zscore'] = (df['ATR14_ratio'] - df['ATR14_ratio'].rolling(20).mean()) / df['ATR14_ratio'].rolling(20).std()
    return df

def compute_volume_flow(close, volume, high, low):
    df = pd.DataFrame(index=close.index)
    avg_vol = volume.rolling(20).mean().replace(0, np.nan)
    df['RVOL20'] = volume / avg_vol
    df['OBV_slope'] = ta.volume.on_balance_volume(close, volume).diff(5) / 5
    df['MFI14'] = ta.volume.money_flow_index(high, low, close, volume, window=14)
    df['dollar_volume_log'] = np.log((close * volume).replace(0, 1))
    df['volume_slope_20'] = volume.diff(5) / 5
    # Thêm cho Risk
    df['OBV_slope_neg'] = (df['OBV_slope'] < 0).astype(int)
    return df

def compute_market_context(close, spy_close):
    df = pd.DataFrame(index=close.index)
    rel_strength = close / spy_close.replace(0, np.nan)
    df['RS_vs_SPY_14'] = rel_strength.pct_change(14)
    df['RS_vs_SPY_30'] = rel_strength.pct_change(30)
    returns = close.pct_change()
    spy_returns = spy_close.pct_change()
    df['beta_60D'] = returns.rolling(60).cov(spy_returns) / spy_returns.rolling(60).var().replace(0, np.nan)
    df['SPY_above_EMA50'] = (spy_close > spy_close.ewm(span=50).mean()).astype(int)
    df['SPY_20d_return'] = spy_close.pct_change(20)
    return df

def build_features(close, high, low, volume, spy_close):
    """Feature chung cho cả Model A và Model C"""
    f_ret = compute_return_dynamics(close)
    f_clv = pd.DataFrame({'CLV': compute_clv(high, low, close)}, index=close.index)
    f_trend = compute_trend_momentum(close)
    f_adx = pd.DataFrame({'ADX14': ta.trend.adx(high, low, close, window=14)}, index=close.index)
    f_break = compute_breakout_positioning(close, high, low)
    f_vol = compute_volatility(close, high, low, volume)
    f_flow = compute_volume_flow(close, volume, high, low)
    f_market = compute_market_context(close, spy_close)
    
    feat = pd.concat([f_ret, f_clv, f_trend, f_adx, f_break, f_vol, f_flow, f_market], axis=1)
    feat['sector_percentile_20d'] = 0.5   # placeholder
    return feat.replace([np.inf, -np.inf], np.nan)

# ==================== 6. Huấn luyện và đánh giá Model A ====================
def train_evaluate_A(X_train, y_train, X_test, y_test, params):
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)
    spearman_corr, _ = spearmanr(y_test, y_pred)
    
    # Daily top decile hit rate
    results = pd.DataFrame({'y_true': y_test.values, 'y_pred': y_pred}, index=X_test.index)
    daily_hits = []
    for date in results.index.get_level_values(0).unique():
        group = results.xs(date, level=0)
        if len(group) < 10:
            continue
        group = group.sort_values('y_pred', ascending=False)
        top_n = max(1, int(0.1 * len(group)))
        top_group = group.head(top_n)
        daily_hits.append((top_group['y_true'] > 0).mean())
    print(f"[Model A] Spearman: {spearman_corr:.4f} | Daily Hit Rate: {np.mean(daily_hits):.4f}")
    return model

# ==================== 7. Huấn luyện Model C (Risk) ====================
def train_evaluate_C(X_train, y_train, X_test, y_test, params):
    # Cân bằng lớp bằng scale_pos_weight
    pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
    params = params.copy()
    params['scale_pos_weight'] = pos_weight
    params['objective'] = 'binary:logistic'
    
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred_prob = model.predict_proba(X_test)[:, 1]
    y_pred_class = (y_pred_prob >= 0.5).astype(int)
    
    from sklearn.metrics import recall_score, precision_score, roc_auc_score
    recall = recall_score(y_test, y_pred_class)
    precision = precision_score(y_test, y_pred_class)
    auc = roc_auc_score(y_test, y_pred_prob)
    print(f"[Model C] Recall (risk=1): {recall:.4f} | Precision: {precision:.4f} | AUC: {auc:.4f}")
    
    # Top-K risk capture: trong top 10% có bao nhiêu % risk thật
    results = pd.DataFrame({'y_true': y_test.values, 'y_prob': y_pred_prob}, index=X_test.index)
    daily_capture = []
    for date in results.index.get_level_values(0).unique():
        group = results.xs(date, level=0)
        if len(group) < 10:
            continue
        group = group.sort_values('y_prob', ascending=False)
        top_n = max(1, int(0.1 * len(group)))
        top_group = group.head(top_n)
        capture = top_group['y_true'].mean()  # tỷ lệ risk thật trong top
        daily_capture.append(capture)
    print(f"[Model C] Top 10% risk capture (daily avg): {np.mean(daily_capture):.4f}")
    return model

# ==================== 8. Backtest kết hợp ====================
def backtest_combined(model_A, model_C, X_test, prices_dict, top_n=10, horizon=14):
    print("\n=== BACKTEST: Model A + Model C (Risk filter) ===")
    results_combined = []
    results_only_A = []
    
    unique_dates = sorted(X_test.index.get_level_values(0).unique())
    for date in unique_dates:
        X_day = X_test.xs(date, level=0)
        if isinstance(X_day, pd.Series):
            X_day = X_day.to_frame().T
        
        # Dự đoán Model A (upside)
        pred_A = model_A.predict(X_day)
        # Dự đoán Model C (risk probability)
        pred_C = model_C.predict_proba(X_day)[:, 1] if hasattr(model_C, 'predict_proba') else model_C.predict(X_day)
        
        # Final score
        final_score = pred_A * (1 - pred_C)
        
        df_pred = pd.DataFrame({
            'Ticker': X_day.index,
            'Pred_A': pred_A,
            'Risk_Prob': pred_C,
            'FinalScore': final_score
        }).sort_values('FinalScore', ascending=False).head(top_n)
        
        # Lấy danh sách top theo FinalScore để giao dịch
        selected = df_pred['Ticker'].tolist()
        
        # Tính lợi nhuận thực tế 14 ngày
        daily_ret_combined = []
        for t in selected:
            try:
                price_series = prices_dict[t]['Close']
                entry_price = price_series.loc[date]
                future_window = price_series.loc[date:].iloc[1:horizon+1]
                if len(future_window) == 0:
                    continue
                exit_price = future_window.max()
                ret = (exit_price - entry_price) / entry_price
                daily_ret_combined.append(ret)
            except:
                continue
        
        if daily_ret_combined:
            results_combined.append(np.mean(daily_ret_combined))
        
        # ---- Đối chứng: chỉ dùng Model A (top theo Pred_A) ----
        df_A_only = pd.DataFrame({'Ticker': X_day.index, 'Pred_A': pred_A}).sort_values('Pred_A', ascending=False).head(top_n)
        selected_A = df_A_only['Ticker'].tolist()
        daily_ret_A = []
        for t in selected_A:
            try:
                price_series = prices_dict[t]['Close']
                entry_price = price_series.loc[date]
                future_window = price_series.loc[date:].iloc[1:horizon+1]
                if len(future_window) == 0:
                    continue
                exit_price = future_window.max()
                ret = (exit_price - entry_price) / entry_price
                daily_ret_A.append(ret)
            except:
                continue
        if daily_ret_A:
            results_only_A.append(np.mean(daily_ret_A))
    
    if len(results_combined) > 0:
        combined_series = pd.Series(results_combined)
        A_series = pd.Series(results_only_A)
        print(f"Model A only : Avg 14D return = {A_series.mean()*100:.2f}% | Win rate = {(A_series>0).mean()*100:.2f}%")
        print(f"Combined (A*C): Avg 14D return = {combined_series.mean()*100:.2f}% | Win rate = {(combined_series>0).mean()*100:.2f}%")
        return combined_series
    else:
        print("Không đủ dữ liệu backtest.")
        return pd.Series()

# ==================== 9. Dự đoán cho ngày hiện tại (kết hợp) ====================
def predict_current_combined(model_A, model_C, tickers):
    print("\nDownloading latest data for prediction...")
    end = datetime.today().strftime('%Y-%m-%d')
    new_data = download_data(tickers, "2023-01-01", end)
    
    # SPY close
    if 'SPY' in new_data.columns.levels[0]:
        spy_close_new = new_data['SPY']['Close']
    else:
        spy_close_new = new_data.iloc[:, 0]
    
    latest_features = []
    latest_tickers = []
    last_date = None
    
    for t in tickers:
        if t == 'SPY' or t not in new_data.columns.levels[0]:
            continue
        df_t = new_data[t].dropna()
        if len(df_t) < 260:
            continue
        close = df_t['Close']
        high = df_t['High']
        low = df_t['Low']
        volume = df_t['Volume']
        feats = build_features(close, high, low, volume, spy_close_new)
        feats = feats.replace([np.inf, -np.inf], np.nan).dropna()
        if feats.empty:
            continue
        last_row = feats.iloc[-1:]
        last_date = last_row.index[0]
        latest_features.append(last_row)
        latest_tickers.append(t)
    
    X_latest = pd.concat(latest_features, axis=0)
    X_latest.index = latest_tickers
    
    print(f"Prediction date: {last_date.date() if last_date else 'unknown'}")
    pred_A = model_A.predict(X_latest) * 100  # %
    pred_C = model_C.predict_proba(X_latest)[:, 1] if hasattr(model_C, 'predict_proba') else model_C.predict(X_latest)
    final_score = pred_A * (1 - pred_C)
    
    result = pd.DataFrame({
        'Ticker': X_latest.index,
        'Upside_%': pred_A,
        'Risk_Prob_%': pred_C * 100,
        'FinalScore': final_score
    }).sort_values('FinalScore', ascending=False)
    
    print("\n=== TOP 20 CANDIDATES (After Risk Filter) ===")
    print(result.head(20))
    return result

def backtest_export_csv_combined(model_A, model_C, X_test, prices_dict,
                                 top_n=10, horizon=14, drawdown_limit=-0.06,
                                 output_csv="all_predictions_combined.csv"):

    print("\n" + "="*80)
    print(f"BACKTEST: MODEL A * (1 - C) | Precision@{top_n} Evaluation")
    print("="*80)

    all_records = []
    daily_precision_scores = []
    all_dates = sorted(X_test.index.get_level_values(0).unique())

    for date in all_dates:
        X_day = X_test.xs(date, level=0)
        if isinstance(X_day, pd.Series): X_day = X_day.to_frame().T
        if len(X_day) < top_n: continue

        # 1. Dự đoán từ Model
        pred_A = model_A.predict(X_day)
        prob_C = model_C.predict_proba(X_day)[:, 1]
        final_score = pred_A * (1 - prob_C)

        df_day = pd.DataFrame({
            'Ticker': X_day.index,
            'Pred_A': pred_A,
            'Risk_Prob': prob_C,
            'FinalScore': final_score
        })

        # 2. Tính toán kết quả thực tế cho TẤT CẢ các mã trong ngày đó để làm Ground Truth
        actual_outcomes = []
        for t in X_day.index:
            try:
                price_series = prices_dict[t]['Close']
                entry_price = price_series.loc[date]
                future_window = price_series.loc[date:].iloc[1:horizon+1]
                
                if len(future_window) < horizon: continue # Đảm bảo đủ cửa sổ quan sát
                
                m_ret = (future_window.max() - entry_price) / entry_price
                m_dd = (future_window.min() - entry_price) / entry_price
                
                actual_outcomes.append({
                    'Ticker': t,
                    'Actual_Max_Ret': m_ret,
                    'Actual_DD': m_dd,
                    # Điều kiện lọc cổ phiếu "Tốt thực sự": Tăng mạnh và không sập sâu
                    'Is_Safe': 1 if m_dd > drawdown_limit else 0
                })
            except: continue
        
        df_actual = pd.DataFrame(actual_outcomes)
        if df_actual.empty: continue

        # Ground Truth: Top N mã có Max Return cao nhất trong số các mã "Safe"
        # Nếu không đủ mã Safe, lấy top mã có DD ít nhất (hoặc tùy biến logic)
        ground_truth_top = df_actual[df_actual['Is_Safe'] == 1].sort_values('Actual_Max_Ret', ascending=False).head(top_n)
        gt_tickers = set(ground_truth_top['Ticker'].tolist())

        # Model Top N: Dựa trên FinalScore
        model_top = df_day.sort_values('FinalScore', ascending=False).head(top_n)
        model_tickers = set(model_top['Ticker'].tolist())

        # 3. Tính Precision@N (Độ trùng lặp)
        hits = len(model_tickers.intersection(gt_tickers))
        precision_n = hits / top_n
        daily_precision_scores.append(precision_n)

        # 4. Lưu log cho các mã Model chọn (để xuất CSV)
        for _, row in model_top.iterrows():
            t = row['Ticker']
            actual_row = df_actual[df_actual['Ticker'] == t]
            if not actual_row.empty:
                res = actual_row.iloc[0]
                all_records.append({
                    'Date': date.date(),
                    'Ticker': t,
                    'Pred_A': round(row['Pred_A'], 4),
                    'Risk_Prob_%': round(row['Risk_Prob'] * 100, 2),
                    'FinalScore': round(row['FinalScore'], 4),
                    'Max_Return_%': round(res['Actual_Max_Ret'] * 100, 2),
                    'Max_Drawdown_%': round(res['Actual_DD'] * 100, 2),
                    'Hit_GT': 1 if t in gt_tickers else 0
                })

    df_final = pd.DataFrame(all_records)
    df_final.to_csv(output_csv, index=False)

    # Thống kê
    avg_precision = np.mean(daily_precision_scores)
    print(f"--- KẾT QUẢ BACKTEST ---")
    print(f"Trung bình Precision@{top_n}: {avg_precision:.2%}")
    print(f"Số ngày giao dịch: {len(daily_precision_scores)}")
    print(f"Tổng số lệnh: {len(df_final)}")
    print(f"Lợi nhuận TB mỗi lệnh (Top {top_n}): {df_final['Max_Return_%'].mean():.2f}%")
    
    return df_final

def evaluate_topn_range(model_A, model_C, X_test, prices_dict,
                        topn_list=range(10, 101, 10),
                        horizon=14):

    summary_rows = []

    for n in topn_list:
        print(f"\n===== Running top_n = {n} =====")

        df = backtest_export_csv_combined(
            model_A,
            model_C,
            X_test,
            prices_dict,
            top_n=n,
            horizon=horizon,
            output_csv=f"tmp_top{n}.csv"
        )

        # Tính return đúng logic: topN mỗi ngày -> mean ngày -> mean toàn kỳ
        daily = (
            df.groupby('Date')['Max_Return_%']
              .mean()
              .reset_index(name='Daily_Return')
        )

        avg_return = daily['Daily_Return'].mean()
        win_rate = df['Win'].mean() * 100
        avg_dd = df['Max_Drawdown_%'].mean()

        summary_rows.append({
            'TopN': n,
            'Trades_per_month~': n * 21,
            'Win_Rate_%': round(win_rate, 2),
            'Avg_14D_Return_%': round(avg_return, 2),
            'Avg_Drawdown_%': round(avg_dd, 2)
        })

    summary = pd.DataFrame(summary_rows)
    print("\n========== TOP N COMPARISON ==========")
    print(summary)

    return summary
    
# ==================== 10. Main ====================
if __name__ == "__main__":
    start_date, end_date = "2019-01-01", "2026-04-30"
    train_end, test_start = "2023-12-31", "2024-01-01"
    
    print("Loading S&P 100...")
    tickers = get_sp100_tickers()
    
    print("Downloading price data...")
    data = download_data(tickers, start_date, end_date)
    
    # SPY
    if 'SPY' in data.columns.levels[0]:
        spy_close = data['SPY']['Close']
    else:
        spy_close = data.iloc[:, 0]
    
    all_X, all_y_A, all_y_C = [], [], []
    
    print("Building features and labels...")
    for t in tickers:
        if t == 'SPY' or t not in data.columns.levels[0]:
            continue
        df_t = data[t].dropna()
        if len(df_t) < 50:
            continue
        close = df_t['Close']
        high = df_t['High']
        low = df_t['Low']
        volume = df_t['Volume']
        
        label_A = compute_label_upside(close, horizon=14)
        label_C = compute_label_risk(close, horizon=14, drawdown_threshold=0.06)
        feats = build_features(close, high, low, volume, spy_close)
        
        common_idx = feats.index.intersection(label_A.index).intersection(label_C.index)
        feats, yA, yC = feats.loc[common_idx], label_A.loc[common_idx], label_C.loc[common_idx]
        valid = ~(feats.isna().any(axis=1) | yA.isna() | yC.isna())
        feats_valid = feats[valid].copy()
        feats_valid['Ticker'] = t
        feats_valid.set_index('Ticker', append=True, inplace=True)
        
        all_X.append(feats_valid)
        all_y_A.append(yA[valid])
        all_y_C.append(yC[valid])
    
    X = pd.concat(all_X, axis=0)
    y_A = pd.concat(all_y_A, axis=0)
    y_C = pd.concat(all_y_C, axis=0)
    
    # Train/test split theo ngày
    train_mask = X.index.get_level_values(0) <= pd.to_datetime(train_end)
    test_mask = X.index.get_level_values(0) >= pd.to_datetime(test_start)
    
    X_train, X_test = X[train_mask], X[test_mask]
    yA_train, yA_test = y_A[train_mask], y_A[test_mask]
    yC_train, yC_test = y_C[train_mask], y_C[test_mask]
    
    print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
    
    # ==================== Tham số cho GPU ====================
    params_A = {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.02,
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda",
        "random_state": 42
    }
    
    params_C = {
        "n_estimators": 150,
        "max_depth": 5,
        "learning_rate": 0.03,
        "tree_method": "hist",
        "device": "cuda",
        "random_state": 42
    }
    
    print("\n--- Training Model A (Upside) ---")
    model_A = train_evaluate_A(X_train, yA_train, X_test, yA_test, params_A)
    
    print("\n--- Training Model C (Risk) ---")
    model_C = train_evaluate_C(X_train, yC_train, X_test, yC_test, params_C)
    
    # Backtest kết hợp
    prices_dict = {t: data[t] for t in tickers if t in data.columns.levels[0]}
    backtest_export_csv_combined(
        model_A, 
        model_C, 
        X_test, 
        prices_dict, 
        top_n=10, 
        horizon=14, 
        drawdown_limit=-0.06
    )
    
    # Dự đoán hiện tại
    predict_current_combined(model_A, model_C, tickers)

    # evaluate_topn_range(
    #     model_A,
    #     model_C,
    #     X_test,
    #     prices_dict,
    #     topn_list=range(10, 101, 10),
    #     horizon=14
    # )
    
    # Feature importance
    imp_A = pd.DataFrame({'feature': X.columns, 'importance': model_A.feature_importances_}).sort_values('importance', ascending=False)
    imp_C = pd.DataFrame({'feature': X.columns, 'importance': model_C.feature_importances_}).sort_values('importance', ascending=False)
    print("\n--- Top 10 features: Model A ---")
    print(imp_A.head(10))
    print("\n--- Top 10 features: Model C ---")
    print(imp_C.head(10))

Loading S&P 100...
Lấy 102 mã S&P 100 (kèm SPY).


[*********************100%***********************]  102 of 102 completed


Building features and labels...
Train size: (100171, 47), Test size: (57159, 47)

--- Training Model A (Upside) ---
[Model A] Spearman: 0.2121 | Daily Hit Rate: 0.8782

--- Training Model C (Risk) ---
[Model C] Recall (risk=1): 0.9204 | Precision: 0.8304 | AUC: 0.9809
[Model C] Top 10% risk capture (daily avg): 0.9452

BACKTEST: MODEL A * (1 - C) | Precision@10 Evaluation
--- KẾT QUẢ BACKTEST ---
Trung bình Precision@10: 19.33%
Số ngày giao dịch: 569
Tổng số lệnh: 5690
Lợi nhuận TB mỗi lệnh (Top 10): 6.85%



[*********************100%***********************]  102 of 102 completed


Prediction date: 2026-05-12

=== TOP 20 CANDIDATES (After Risk Filter) ===
    Ticker   Upside_%  Risk_Prob_%  FinalScore
72     NOW  12.848440    17.346275   10.619715
4     ADBE   9.625965     4.710270    9.172556
71     NKE   9.417359     3.288973    9.107625
48    INTU   9.764664     7.728125    9.010038
3      ACN   9.077032     4.478187    8.670546
26     CRM   8.173488     5.429463    7.729711
78    PLTR   7.282690     8.983791    6.628428
24     COP   6.879667     4.478513    6.571560
76     PFE   5.729570     1.395654    5.649605
100    XOM   5.634139     3.282164    5.449217
23     COF   5.491064     2.007214    5.380846
31     DHR   5.518377     2.734768    5.367463
15    BKNG   5.624061     6.286774    5.270489
9     AMZN   5.515334     5.855467    5.192386
22   CMCSA   5.279147     4.839396    5.023668
46     IBM   5.211698     3.764053    5.015527
70    NFLX   5.192140     3.985554    4.985205
2      ABT   5.160933     3.622040    4.974002
19       C   4.918501     1.3502